
# 07. Embeddings and Explainable NLP

## Embedding theory
Word embeddings map tokens into dense vectors where semantic similarity becomes geometric proximity.

## Methods implemented
- Trainable neural embeddings (inside language models)
- Word2Vec (gensim)
- FastText (gensim, subword-aware)

## Explainability in this notebook
- Embedding neighborhood analysis
- PCA / t-SNE / UMAP projections
- Transformer attention heatmaps


In [ ]:

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from utils.data import prepare_combined_corpus
from utils.embeddings import embedding_dataframe, nearest_neighbors, train_fasttext, train_word2vec
from utils.tokenization import NLTKTokenizerBackend, normalize_text

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()

bundle = prepare_combined_corpus(
    project_root=PROJECT_ROOT,
    include_wikitext=True,
    wikitext_train_tokens=120_000,
    wikitext_val_tokens=20_000,
    wikitext_test_tokens=20_000,
)

sentences = normalize_text(bundle.train_text).split(".")
tokenized = [NLTKTokenizerBackend().tokenize(sent) for sent in sentences]
tokenized = [s for s in tokenized if len(s) > 3][:8000]

w2v = train_word2vec(tokenized, vector_size=100, epochs=5, min_count=2)
ft = train_fasttext(tokenized, vector_size=100, epochs=5, min_count=2)

print("Word2Vec neighbors for 'time':", nearest_neighbors(w2v.wv, "time", topn=5))
print("FastText neighbors for 'time':", nearest_neighbors(ft.wv, "time", topn=5))


In [ ]:

pca_df = embedding_dataframe(w2v.wv, method="pca", max_words=250)

plt.figure(figsize=(8, 6))
plt.scatter(pca_df["x"], pca_df["y"], s=8, alpha=0.7)
for _, row in pca_df.head(40).iterrows():
    plt.text(row["x"], row["y"], row["word"], fontsize=7)
plt.title("Word2Vec PCA projection")
plt.show()


In [ ]:

tsne_df = embedding_dataframe(w2v.wv, method="tsne", max_words=250)
plt.figure(figsize=(8, 6))
plt.scatter(tsne_df["x"], tsne_df["y"], s=8, alpha=0.7, color="#a855f7")
plt.title("Word2Vec t-SNE projection")
plt.show()


In [ ]:

umap_df = embedding_dataframe(w2v.wv, method="umap", max_words=250)
plt.figure(figsize=(8, 6))
plt.scatter(umap_df["x"], umap_df["y"], s=8, alpha=0.7, color="#0ea5e9")
plt.title("Word2Vec UMAP projection")
plt.show()



## Interpretation
FastText tends to perform better on rare or morphologically related words due to subword modeling.
Embedding projections are approximate but useful for cluster-level interpretation.
